In [ ]:
import requests
import pandas as pd
from collections import defaultdict

In [ ]:
dataset_url = "https://api.census.gov/data/2024/acs/acs5" # 2024
api_key = "32685a5c4ce6a6c58598b54cce7cf23a6608f8f0"

metadata_url = f"{dataset_url}/variables.json"
meta_response = requests.get(metadata_url)
variables_meta = meta_response.json()["variables"]
var_label_map = {
    var_id: meta["label"].replace("!!", " ")
    for var_id, meta in variables_meta.items()
    if "label" in meta
}

race_api = {
    'B01001A': 'White Alone',
    'B01001B': 'Black or African American Alone',
    'B01001C': 'American Indian and Alaska Native Alone',
    'B01001D': 'Asian Alone',
    'B01001E': 'Native Hawaiian and Other Pacific Islander Alone',
    'B01001F': 'Some Other Race Alone',
    'B01001G': 'Two or More Races',
    'B01001H': 'White Alone, Not Hispanic or Latino',
    'B01001I': 'Hispanic / Latino (Any race)',
}
# params for loading
# for: state:26 is michigan
# do: for: us-1 for USA
params = {
    "get": "group(EEOALL1R)",
    "EEO": "0000",
    "ucgid": "0400000US26",
    "key": api_key
}
data_response = requests.get(dataset_url, params=params)
raw_data = data_response.json()

df = None

for group_id in race_api:
    params = {
        "get": f"group({group_id})",
        "for": "state:26",
        "key": api_key,
    }

    response = requests.get(dataset_url, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()

    group_df = pd.DataFrame(data[1:], columns=data[0])

    # Keep geographic identifiers and this race table's variables.
    columns = ["NAME", "state"] + [
        col for col in group_df.columns
        if col.startswith(f"{group_id}_")
    ]
    group_df = group_df[columns]

    if df is None:
        df = group_df
    else:
        df = df.merge(
            group_df,
            on=["NAME", "state"],
            validate="one_to_one",
        )

df

In [ ]:
group_meta = {}

for group_id in race_api:
    group_response = requests.get(
        f"{dataset_url}/groups/{group_id}.json",
        timeout=30,
    )
    group_response.raise_for_status()
    group_meta.update(group_response.json()["variables"])

cleaned_data_dict = defaultdict(list)
cleaned_data_columns = list()

for column, value in df.iloc[0].items():
    label = group_meta.get(column, {}).get("label", column)
    label = label.replace("!!", " ").replace(":", "")

    table_id = column.split("_")[0]
    race = race_api.get(table_id)

    if race:
        if "Annotation" not in label:
    #         print(f"{race} - {label}: {value}")
            cleaned_data_dict[race].append(value)
            if label not in cleaned_data_columns:
                cleaned_data_columns.append(label)
    # else:
    #     print(f"{label}: {value}")

In [ ]:
for col in df:
    print(col)